In [1]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 192) / Test: (90067, 191)


In [2]:
# feature_refinement 재적용 + 피처 준비
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)

print('피처 정제 완료')

피처 정제 완료


In [3]:
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

# LightGBM용 category dtype
X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

# CatBoost용 인덱스
cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

N_SPLITS = 5
SEED     = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

피처: 197개 / 범주형: 21개
scale_pos_weight: 2.87


In [4]:
# LightGBM optuna 탐색
def lgb_objective(trial):
    params = {
        'objective':          'binary',
        'metric':             'auc',
        'boosting_type':      'gbdt',
        'num_leaves':         trial.suggest_int('num_leaves', 20, 150),
        'max_depth':          trial.suggest_int('max_depth', 4, 8),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'n_estimators':       trial.suggest_int('n_estimators', 300, 1000),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda':         trial.suggest_float('reg_lambda', 0.0, 5.0),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'scale_pos_weight':   spw,
        'random_state':       SEED,
        'verbose':            -1,
    }

    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    fold_aucs = []

    for fold, (tr_idx, val_idx) in enumerate(skf3.split(X_lgb, y)):
        X_tr  = X_lgb.iloc[tr_idx]
        X_val = X_lgb.iloc[val_idx]
        y_tr  = y.iloc[tr_idx]
        y_val = y.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(30, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        fold_aucs.append(roc_auc_score(y_val, val_pred))

        trial.report(np.mean(fold_aucs), fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_aucs)

print('LightGBM Objective 정의 완료')

LightGBM Objective 정의 완료


In [5]:
sampler = optuna.samplers.TPESampler(seed=SEED)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)

lgb_study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
)

def print_callback(study, trial):
    if trial.value is None:
        return
    print(f'Trial {trial.number:3d} | AUC: {trial.value:.5f} | Best: {study.best_value:.5f} | leaves={trial.params.get("num_leaves")}, lr={trial.params.get("learning_rate"):.4f}, depth={trial.params.get("max_depth")}')

lgb_study.optimize(
    lgb_objective,
    n_trials=50,
    show_progress_bar=False,
    callbacks=[print_callback],
)

print(f'\n✅ 최적 AUC (3-fold): {lgb_study.best_value:.5f}')
print('최적 파라미터:')
for k, v in lgb_study.best_params.items():
    print(f'  {k}: {v}')

Trial   0 | AUC: 0.73891 | Best: 0.73891 | leaves=69, lr=0.0540, depth=8
Trial   1 | AUC: 0.73943 | Best: 0.73943 | leaves=112, lr=0.0933, depth=4
Trial   2 | AUC: 0.73931 | Best: 0.73943 | leaves=76, lr=0.0409, depth=5
Trial   3 | AUC: 0.73949 | Best: 0.73949 | leaves=87, lr=0.0111, depth=6
Trial   4 | AUC: 0.73944 | Best: 0.73949 | leaves=59, lr=0.0483, depth=4
Trial   5 | AUC: 0.73951 | Best: 0.73951 | leaves=106, lr=0.0331, depth=5
Trial   6 | AUC: 0.73897 | Best: 0.73951 | leaves=98, lr=0.0123, depth=8
Trial   7 | AUC: 0.73939 | Best: 0.73951 | leaves=66, lr=0.0349, depth=5
Trial   8 | AUC: 0.73957 | Best: 0.73957 | leaves=20, lr=0.0509, depth=8
Trial   9 | AUC: 0.73873 | Best: 0.73957 | leaves=101, lr=0.0116, depth=5
Trial  10 | AUC: 0.73968 | Best: 0.73968 | leaves=22, lr=0.0203, depth=7
Trial  11 | AUC: 0.73964 | Best: 0.73968 | leaves=20, lr=0.0192, depth=7
Trial  12 | AUC: 0.73955 | Best: 0.73968 | leaves=21, lr=0.0194, depth=7
Trial  13 | AUC: 0.73918 | Best: 0.73968 | leave

In [6]:
# 최적 파라미터로 LightGBM 5-fold 최종 학습
best_lgb_params = lgb_study.best_params
best_lgb_params.update({
    'objective':        'binary',
    'metric':           'auc',
    'boosting_type':    'gbdt',
    'scale_pos_weight': spw,
    'random_state':     SEED,
    'verbose':         -1,
})

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
lgb_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgb, y)):
    X_tr  = X_lgb.iloc[tr_idx]
    X_val = X_lgb.iloc[val_idx]
    y_tr  = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    model = lgb.LGBMClassifier(**best_lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(-1)
        ]
    )

    val_pred = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, val_pred)

    oof_lgb[val_idx]  = val_pred
    test_lgb         += model.predict_proba(X_test_lgb)[:, 1] / N_SPLITS
    lgb_fold_scores.append(fold_auc)
    print(f'[LightGBM] Fold {fold+1} | AUC: {fold_auc:.5f} | Best iter: {model.best_iteration_}')

lgb_auc = roc_auc_score(y, oof_lgb)
print(f'\n✅ LightGBM OOF AUC (튜닝 후): {lgb_auc:.5f} ± {np.std(lgb_fold_scores):.5f}')
print(f'   튜닝 전 LightGBM : 0.73926')
print(f'   개선폭            : {lgb_auc - 0.73926:+.5f}')

[LightGBM] Fold 1 | AUC: 0.73788 | Best iter: 483
[LightGBM] Fold 2 | AUC: 0.74207 | Best iter: 624
[LightGBM] Fold 3 | AUC: 0.74041 | Best iter: 444
[LightGBM] Fold 4 | AUC: 0.73825 | Best iter: 455
[LightGBM] Fold 5 | AUC: 0.74038 | Best iter: 429

✅ LightGBM OOF AUC (튜닝 후): 0.73978 ± 0.00155
   튜닝 전 LightGBM : 0.73926
   개선폭            : +0.00052


In [7]:
# CatBoost 재학습 + 앙상블 가중치 재탐색
# CatBoost Optuna 최적 파라미터
cat_params = {
    'iterations':            794,
    'learning_rate':         0.030094391673729362,
    'depth':                 7,
    'l2_leaf_reg':           7.433949857398995,
    'bagging_temperature':   0.3980358270898108,
    'random_strength':       1.3075975210328405,
    'border_count':          108,
    'loss_function':         'Logloss',
    'eval_metric':           'AUC',
    'scale_pos_weight':      spw,
    'random_seed':           SEED,
    'early_stopping_rounds': 50,
    'verbose':               False,
    'use_best_model':        True,
}

oof_cat  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))
cat_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr  = X.iloc[tr_idx].reset_index(drop=True)
    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_tr  = y.iloc[tr_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    train_pool = Pool(X_tr, y_tr, cat_features=cat_feature_indices)
    val_pool   = Pool(X_val, y_val, cat_features=cat_feature_indices)
    test_pool  = Pool(X_test, cat_features=cat_feature_indices)

    model = CatBoostClassifier(**cat_params)
    model.fit(train_pool, eval_set=val_pool)

    val_pred = model.predict_proba(val_pool)[:, 1]
    fold_auc = roc_auc_score(y_val, val_pred)

    oof_cat[val_idx]  = val_pred
    test_cat         += model.predict_proba(test_pool)[:, 1] / N_SPLITS
    cat_fold_scores.append(fold_auc)
    print(f'[CatBoost] Fold {fold+1} | AUC: {fold_auc:.5f}')

cat_auc = roc_auc_score(y, oof_cat)
print(f'\n✅ CatBoost OOF AUC: {cat_auc:.5f}')

[CatBoost] Fold 1 | AUC: 0.73848
[CatBoost] Fold 2 | AUC: 0.74320
[CatBoost] Fold 3 | AUC: 0.74083
[CatBoost] Fold 4 | AUC: 0.73832
[CatBoost] Fold 5 | AUC: 0.74077

✅ CatBoost OOF AUC: 0.74031


In [8]:
# 최적 가중치 재탐색
best_w, best_auc = 0.5, 0.0
results = []

for w_cat in np.arange(0.0, 1.05, 0.05):
    w_lgb = 1 - w_cat
    blended = oof_cat * w_cat + oof_lgb * w_lgb
    auc = roc_auc_score(y, blended)
    results.append((round(w_cat, 2), round(w_lgb, 2), auc))
    if auc > best_auc:
        best_auc = auc
        best_w   = w_cat

results_df = pd.DataFrame(results, columns=['w_catboost','w_lgbm','oof_auc'])
print('=== 가중치별 OOF AUC ===')
display(results_df.sort_values('oof_auc', ascending=False).head(10))

print(f'\n✅ CatBoost  : {cat_auc:.5f}')
print(f'✅ LightGBM  : {lgb_auc:.5f}')
print(f'✅ 앙상블     : {best_auc:.5f}  (w_cat={best_w:.2f} : w_lgb={1-best_w:.2f})')
print(f'이전 앙상블   : 0.74035')
print(f'베이스라인    : 0.74008')
print(f'개선폭        : {best_auc - 0.74008:+.5f}')

=== 가중치별 OOF AUC ===


,w_catboost,w_lgbm,oof_auc
14,0.70,0.30,0.740411
15,0.75,0.25,0.740409
13,0.65,0.35,0.740405
16,0.80,0.20,0.740402
12,0.60,0.40,0.740395
17,0.85,0.15,0.740389
11,0.55,0.45,0.740378
18,0.90,0.10,0.740370
10,0.50,0.50,0.740355
19,0.95,0.05,0.740345



✅ CatBoost  : 0.74031
✅ LightGBM  : 0.73978
✅ 앙상블     : 0.74041  (w_cat=0.70 : w_lgb=0.30)
이전 앙상블   : 0.74035
베이스라인    : 0.74008
개선폭        : +0.00033


In [ ]:
# 제출 파일 생성
test_blended = test_cat * best_w + test_lgb * (1 - best_w)

sample_sub = pd.read_csv('/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw/sample_submission.csv')  # 경로 본인 환경에 맞게 수정
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_blended

submission.to_csv('submission_ensemble_lgbm_optuna.csv', index=False, encoding='utf-8-sig')

print('저장 완료: submission_ensemble_lgbm_optuna.csv')
print(f'\n최종 OOF AUC 요약')
print(f'  CatBoost (튜닝)    : {cat_auc:.5f}')
print(f'  LightGBM (튜닝)    : {lgb_auc:.5f}')
print(f'  앙상블              : {best_auc:.5f}  (w_cat={best_w:.2f}, w_lgb={1-best_w:.2f})')
print(f'  이전 앙상블         : 0.74035')
print(f'  베이스라인          : 0.74008')
print(f'  총 개선폭           : {best_auc - 0.74008:+.5f}')
display(submission.head())

FileNotFoundError: [Errno 2] No such file or directory: 'sample_submission.csv'